In [2]:
# 08_previous_day_rolling_trend_feature_engineering.ipynb
# Cell 1. 이전일(previous-day) 데이터와 daytime_only feature 목록을 로드하고 기본 유효성을 점검합니다.

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

# 노트북을 notebooks/ 안에서 실행해도, 프로젝트 루트에서 실행해도 동작하도록 경로를 잡습니다.
CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_PATH = PROJECT_DIR / "data/processed/deep_learning_previous_day/modeling_dataset_previous_day_encoded.csv"
DAYTIME_FEATURE_PATH = PROJECT_DIR / "data/processed/deep_learning_feature_subsets/daytime_only_features.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_PATH exists:", DATA_PATH.exists())
print("DAYTIME_FEATURE_PATH exists:", DAYTIME_FEATURE_PATH.exists())

# CSV는 UTF-8 계열로 읽습니다. utf-8-sig는 BOM이 있어도 안전하게 처리합니다.
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
daytime_feature_table = pd.read_csv(DAYTIME_FEATURE_PATH, encoding="utf-8-sig")

daytime_features = daytime_feature_table["feature"].dropna().astype(str).tolist()

print("\n=== Loaded Shapes ===")
print("previous-day dataset:", df.shape)
print("daytime_only feature table:", daytime_feature_table.shape)
print("daytime_only feature count:", len(daytime_features))

print("\n=== Key Columns ===")
key_columns = [
    "participant_object_id",
    "calendar_date",
    "feature_date",
    "feature_lag_days",
    "good_sleep_label",
    "deep_learning_split",
]
display(df[key_columns].head())

print("\n=== Daytime Feature List ===")
display(daytime_feature_table)

missing_daytime_features = sorted(set(daytime_features) - set(df.columns))
extra_daytime_features = sorted(set(daytime_features) - set(daytime_feature_table["feature"]))

print("\n=== Feature Availability Check ===")
print("missing daytime_only features in dataset:", len(missing_daytime_features))
if missing_daytime_features:
    display(pd.DataFrame({"missing_feature": missing_daytime_features}))

# 날짜와 lag를 명시적으로 검증합니다.
df["calendar_date"] = pd.to_datetime(df["calendar_date"], errors="coerce")
df["feature_date"] = pd.to_datetime(df["feature_date"], errors="coerce")
df["computed_lag_days"] = (df["calendar_date"] - df["feature_date"]).dt.days

lag_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "calendar_date_missing",
            "feature_date_missing",
            "feature_lag_days_missing",
            "computed_lag_days_missing",
            "feature_lag_days_equals_1_rows",
            "computed_lag_days_equals_1_rows",
            "both_lag_checks_equal_1_rows",
        ],
        "value": [
            len(df),
            int(df["calendar_date"].isna().sum()),
            int(df["feature_date"].isna().sum()),
            int(df["feature_lag_days"].isna().sum()),
            int(df["computed_lag_days"].isna().sum()),
            int((df["feature_lag_days"] == 1).sum()),
            int((df["computed_lag_days"] == 1).sum()),
            int(((df["feature_lag_days"] == 1) & (df["computed_lag_days"] == 1)).sum()),
        ],
    }
)

print("\n=== Lag Validity Summary ===")
display(lag_summary)

print("\nfeature_lag_days distribution:")
display(df["feature_lag_days"].value_counts(dropna=False).sort_index().rename("rows").to_frame())

print("\ncomputed_lag_days distribution:")
display(df["computed_lag_days"].value_counts(dropna=False).sort_index().rename("rows").to_frame())

invalid_lag_rows = df.loc[
    (df["feature_lag_days"] != 1) | (df["computed_lag_days"] != 1),
    key_columns + ["computed_lag_days"],
]

print("\ninvalid lag rows:", len(invalid_lag_rows))
if len(invalid_lag_rows) > 0:
    display(invalid_lag_rows.head(20))

# split별 row 수, participant 수, target 분포를 확인합니다.
split_counts = (
    df.groupby("deep_learning_split", dropna=False)
    .agg(
        rows=("good_sleep_label", "size"),
        participants=("participant_object_id", "nunique"),
        target_mean=("good_sleep_label", "mean"),
        target_0=("good_sleep_label", lambda s: int((s == 0).sum())),
        target_1=("good_sleep_label", lambda s: int((s == 1).sum())),
    )
    .reset_index()
)

print("\n=== Split Counts And Target Distribution ===")
display(split_counts)

overall_target = (
    df["good_sleep_label"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("good_sleep_label")
    .reset_index(name="rows")
)
overall_target["ratio"] = overall_target["rows"] / overall_target["rows"].sum()

print("\n=== Overall Target Distribution ===")
display(overall_target)

# 참가자 split overlap이 있으면 participant-aware split 해석에 문제가 생기므로 먼저 확인합니다.
participants_by_split = {
    split_name: set(participants)
    for split_name, participants in df.groupby("deep_learning_split")["participant_object_id"]
}

overlap_rows = []
for left_split, left_participants in participants_by_split.items():
    for right_split, right_participants in participants_by_split.items():
        if left_split >= right_split:
            continue
        overlap = sorted(left_participants & right_participants)
        overlap_rows.append(
            {
                "split_pair": f"{left_split} vs {right_split}",
                "overlap_participants": len(overlap),
                "example_participants": overlap[:5],
            }
        )

print("\n=== Participant Split Overlap Check ===")
display(pd.DataFrame(overlap_rows))

# 다음 셀에서 rolling/trend feature를 만들 때 사용할 기본 feature matrix 후보를 미리 점검합니다.
daytime_df = df[key_columns + daytime_features].copy()

numeric_daytime_check = daytime_df[daytime_features].select_dtypes(include=[np.number]).columns.tolist()
non_numeric_daytime_features = sorted(set(daytime_features) - set(numeric_daytime_check))

print("\n=== Daytime Feature Numeric Check ===")
print("numeric daytime_only features:", len(numeric_daytime_check))
print("non-numeric daytime_only features:", len(non_numeric_daytime_features))
if non_numeric_daytime_features:
    display(pd.DataFrame({"non_numeric_feature": non_numeric_daytime_features}))

daytime_missing_summary = (
    daytime_df[daytime_features]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "feature"})
)

print("\n=== Daytime Feature Missing Rate ===")
display(daytime_missing_summary)

print("\nReady for next step: build previous-day daytime_only rolling/trend features.")

PROJECT_DIR: c:\workSpace\DeepLearnin_sleep
DATA_PATH exists: True
DAYTIME_FEATURE_PATH exists: True

=== Loaded Shapes ===
previous-day dataset: (3116, 203)
daytime_only feature table: (19, 2)
daytime_only feature count: 19

=== Key Columns ===


,participant_object_id,calendar_date,feature_date,feature_lag_days,good_sleep_label,deep_learning_split
0,621e2e8e67b776a24055b564,2021-05-25,2021-05-24,1.0,1,train
1,621e2e8e67b776a24055b564,2021-05-26,2021-05-25,1.0,1,train
2,621e2e8e67b776a24055b564,2021-05-27,2021-05-26,1.0,1,train
3,621e2e8e67b776a24055b564,2021-05-28,2021-05-27,1.0,1,train
4,621e2e8e67b776a24055b564,2021-05-29,2021-05-28,1.0,1,train



=== Daytime Feature List ===


,feature,risk_group
0,resting_hr_resting_heart_rate_mean,medium_risk_resting_hr
1,resting_hr_error_mean,medium_risk_resting_hr
2,resting_hr_record_count,medium_risk_resting_hr
3,lightly_active_minutes_sum_missing_ind,medium_risk_same_date_daytime__missing_indicator
4,lightly_active_minutes_sum,medium_risk_same_date_daytime
5,moderately_active_minutes_sum_missing_ind,medium_risk_same_date_daytime__missing_indicator
6,moderately_active_minutes_sum,medium_risk_same_date_daytime
7,sedentary_minutes_sum_missing_ind,medium_risk_same_date_daytime__missing_indicator
8,sedentary_minutes_sum,medium_risk_same_date_daytime
9,very_active_minutes_sum_missing_ind,medium_risk_same_date_daytime__missing_indicator



=== Feature Availability Check ===
missing daytime_only features in dataset: 0

=== Lag Validity Summary ===


...


=== Participant Split Overlap Check ===


,split_pair,overlap_participants,example_participants
0,test vs train,0,[]
1,test vs validation,0,[]
2,train vs validation,0,[]



=== Daytime Feature Numeric Check ===
numeric daytime_only features: 19
non-numeric daytime_only features: 0

=== Daytime Feature Missing Rate ===


,feature,missing_rate
0,resting_hr_resting_heart_rate_mean,0.0
1,resting_hr_error_mean,0.0
2,resting_hr_record_count,0.0
3,lightly_active_minutes_sum_missing_ind,0.0
4,lightly_active_minutes_sum,0.0
5,moderately_active_minutes_sum_missing_ind,0.0
6,moderately_active_minutes_sum,0.0
7,sedentary_minutes_sum_missing_ind,0.0
8,sedentary_minutes_sum,0.0
9,very_active_minutes_sum_missing_ind,0.0



Ready for next step: build previous-day daytime_only rolling/trend features.


In [3]:
# Cell 2. daytime_only previous-day feature에서 rolling / trend 후보 feature를 만든다.
# 목적:
# - target date 기준으로 이미 previous-day feature만 들어온 상태다.
# - 따라서 각 row의 원본 feature는 target_date - 1일 정보다.
# - rolling history는 participant별 feature_date 순서에서 현재 previous-day row까지 포함한다.
# - 아직 파일 저장은 하지 않고, feature 수와 결측 발생만 점검한다.

from pandas.api.types import is_numeric_dtype

# 이전 셀 변수명을 유연하게 받습니다.
if "previous_df" in globals():
    base_df = previous_df.copy()
elif "df" in globals():
    base_df = df.copy()
else:
    raise NameError("previous_df 또는 df가 필요합니다. 1번 셀을 먼저 실행해 주세요.")

if "daytime_only_features" in globals():
    base_features = list(daytime_only_features)
elif "daytime_features" in globals():
    base_features = list(daytime_features)
else:
    raise NameError("daytime_only_features 또는 daytime_features가 필요합니다. 1번 셀을 먼저 실행해 주세요.")

ID_COL = "participant_object_id"
DATE_COL = "calendar_date"
FEATURE_DATE_COL = "feature_date"
TARGET_COL = "good_sleep_label"
SPLIT_COL = "deep_learning_split"

base_df[DATE_COL] = pd.to_datetime(base_df[DATE_COL])
base_df[FEATURE_DATE_COL] = pd.to_datetime(base_df[FEATURE_DATE_COL])
base_df = base_df.sort_values([ID_COL, FEATURE_DATE_COL, DATE_COL]).reset_index(drop=True)

non_numeric_features = [
    col for col in base_features
    if col not in base_df.columns or not is_numeric_dtype(base_df[col])
]

print("base_df shape:", base_df.shape)
print("base daytime features:", len(base_features))
print("non-numeric or missing base features:", len(non_numeric_features))
if non_numeric_features:
    display(pd.DataFrame({"feature": non_numeric_features}))

rolling_source_features = [col for col in base_features if col not in non_numeric_features]

# rolling std는 window 초반에 NaN이 생기므로, min_periods=1 후 std NaN은 0으로 채운다.
# history가 1개뿐이면 변동성이 없다고 보는 보수적 처리다.
ROLLING_WINDOWS = [3, 7]
ROLLING_STATS = ["mean", "std", "min", "max"]

engineered_df = base_df.copy()
created_features = []

grouped = engineered_df.groupby(ID_COL, sort=False)

for window in ROLLING_WINDOWS:
    for feature in rolling_source_features:
        rolling_obj = grouped[feature].rolling(window=window, min_periods=1)

        mean_col = f"{feature}_roll{window}_mean"
        std_col = f"{feature}_roll{window}_std"
        min_col = f"{feature}_roll{window}_min"
        max_col = f"{feature}_roll{window}_max"

        engineered_df[mean_col] = rolling_obj.mean().reset_index(level=0, drop=True)
        engineered_df[std_col] = rolling_obj.std().reset_index(level=0, drop=True).fillna(0.0)
        engineered_df[min_col] = rolling_obj.min().reset_index(level=0, drop=True)
        engineered_df[max_col] = rolling_obj.max().reset_index(level=0, drop=True)

        created_features.extend([mean_col, std_col, min_col, max_col])

# 1일 변화량: 현재 previous-day feature와 바로 이전 feature_date row의 차이.
# participant별 첫 row는 과거가 없으므로 0으로 채운다.
for feature in rolling_source_features:
    diff1_col = f"{feature}_diff1"
    engineered_df[diff1_col] = grouped[feature].diff(1).fillna(0.0)
    created_features.append(diff1_col)

# 3일/7일 rolling 평균 대비 현재값의 편차.
# 단순 diff보다 "최근 평균보다 높은지/낮은지"를 보는 trend 후보로 사용한다.
for window in ROLLING_WINDOWS:
    for feature in rolling_source_features:
        mean_col = f"{feature}_roll{window}_mean"
        dev_col = f"{feature}_dev_from_roll{window}_mean"
        engineered_df[dev_col] = engineered_df[feature] - engineered_df[mean_col]
        created_features.append(dev_col)

# feature_date 간격이 끊기는 구간을 표시한다.
# 다음 sequence/tensor 생성에서 contiguous history 조건을 어떻게 둘지 판단하기 위한 진단 feature다.
engineered_df["previous_feature_date"] = grouped[FEATURE_DATE_COL].shift(1)
engineered_df["feature_date_gap_days"] = (
    engineered_df[FEATURE_DATE_COL] - engineered_df["previous_feature_date"]
).dt.days
engineered_df["feature_date_gap_days"] = engineered_df["feature_date_gap_days"].fillna(0).astype(int)

print("\n=== Engineered Feature Summary ===")
print("original dataframe shape:", base_df.shape)
print("engineered dataframe shape:", engineered_df.shape)
print("created engineered features:", len(created_features))
print("total candidate features:", len(base_features) + len(created_features))

print("\n=== Engineered Feature Examples ===")
display(pd.DataFrame({"created_feature": created_features[:40]}))

engineered_missing = (
    engineered_df[created_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "feature"})
)

print("\n=== Missing Check For Engineered Features ===")
display(engineered_missing.head(30))
print("engineered features with any missing:", int((engineered_missing["missing_count"] > 0).sum()))

engineered_inf_counts = np.isinf(engineered_df[created_features].to_numpy(dtype=float)).sum()
print("engineered Inf count:", int(engineered_inf_counts))

print("\n=== Feature Date Gap Distribution ===")
display(
    engineered_df["feature_date_gap_days"]
    .value_counts(dropna=False)
    .sort_index()
    .rename("rows")
    .reset_index()
    .rename(columns={"index": "feature_date_gap_days"})
)

print("\n=== Split-Level Row/Target Check After Feature Engineering ===")
display(
    engineered_df
    .groupby(SPLIT_COL)
    .agg(
        rows=(TARGET_COL, "size"),
        participants=(ID_COL, "nunique"),
        target_mean=(TARGET_COL, "mean"),
        min_target_date=(DATE_COL, "min"),
        max_target_date=(DATE_COL, "max"),
    )
    .reset_index()
)

print("\nReady for next step: decide feature set, optional calendar features, and tensor creation rules.")

base_df shape: (3116, 203)
base daytime features: 19
non-numeric or missing base features: 0


C:\Users\human-23\AppData\Local\Temp\ipykernel_5232\3119201419.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  engineered_df[mean_col] = rolling_obj.mean().reset_index(level=0, drop=True)
C:\Users\human-23\AppData\Local\Temp\ipykernel_5232\3119201419.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  engineered_df[std_col] = rolling_obj.std().reset_index(level=0, drop=True).fillna(0.0)
C:\Users\human-23\AppData\Local\Temp\ipykernel_5232\3119201419.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually


=== Engineered Feature Summary ===
original dataframe shape: (3116, 203)
engineered dataframe shape: (3116, 414)
created engineered features: 209
total candidate features: 228

=== Engineered Feature Examples ===


C:\Users\human-23\AppData\Local\Temp\ipykernel_5232\3119201419.py:87: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  engineered_df[dev_col] = engineered_df[feature] - engineered_df[mean_col]
C:\Users\human-23\AppData\Local\Temp\ipykernel_5232\3119201419.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  engineered_df["previous_feature_date"] = grouped[FEATURE_DATE_COL].shift(1)
C:\Users\human-23\AppData\Local\Temp\ipykernel_5232\3119201419.py:93: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of

,created_feature
0,resting_hr_resting_heart_rate_mean_roll3_mean
1,resting_hr_resting_heart_rate_mean_roll3_std
2,resting_hr_resting_heart_rate_mean_roll3_min
3,resting_hr_resting_heart_rate_mean_roll3_max
4,resting_hr_error_mean_roll3_mean
5,resting_hr_error_mean_roll3_std
6,resting_hr_error_mean_roll3_min
7,resting_hr_error_mean_roll3_max
8,resting_hr_record_count_roll3_mean
9,resting_hr_record_count_roll3_std


...

engineered features with any missing: 0
engineered Inf count: 0

=== Feature Date Gap Distribution ===


,feature_date_gap_days,rows
0,0,63
1,1,2797
2,3,142
3,4,28
4,5,21
5,6,8
6,7,11
7,8,6
8,9,4
9,10,3



=== Split-Level Row/Target Check After Feature Engineering ===


,deep_learning_split,rows,participants,target_mean,min_target_date,max_target_date
0,test,524,12,0.379771,2021-05-25,2022-01-22
1,train,2135,41,0.405621,2021-05-25,2022-01-21
2,validation,457,10,0.393873,2021-05-25,2022-01-21



Ready for next step: decide feature set, optional calendar features, and tensor creation rules.


In [4]:
# Cell 3. 날짜 공백을 넘지 않는 block-aware rolling/trend feature 생성
# 원하는 결과:
# - gap이 있는 구간을 넘어서 rolling history가 섞이지 않도록 contiguous block을 만든다.
# - 3일/7일 rolling mean/std/min/max와 diff/trend를 block 안에서만 계산한다.
# - MLP current-day용 row 수와 GRU용 contiguous window 후보 sample 수를 미리 확인한다.
# - 아직 파일 저장은 하지 않는다.

from collections import defaultdict

ID_COL = "participant_object_id"
DATE_COL = "calendar_date"
FEATURE_DATE_COL = "feature_date"
TARGET_COL = "good_sleep_label"
SPLIT_COL = "deep_learning_split"

# 2번 셀에서 만든 base_df/base_features를 우선 사용합니다.
# 없으면 1번 셀 변수명에서 다시 가져옵니다.
if "base_df" not in globals():
    if "previous_df" in globals():
        base_df = previous_df.copy()
    elif "df" in globals():
        base_df = df.copy()
    else:
        raise NameError("base_df, previous_df, df 중 하나가 필요합니다.")

if "base_features" not in globals():
    if "daytime_only_features" in globals():
        base_features = list(daytime_only_features)
    elif "daytime_features" in globals():
        base_features = list(daytime_features)
    else:
        raise NameError("base_features 또는 daytime_only_features가 필요합니다.")

block_df = base_df.copy()
block_df[DATE_COL] = pd.to_datetime(block_df[DATE_COL])
block_df[FEATURE_DATE_COL] = pd.to_datetime(block_df[FEATURE_DATE_COL])
block_df = block_df.sort_values([ID_COL, FEATURE_DATE_COL, DATE_COL]).reset_index(drop=True)

# participant별 feature_date 간격을 계산합니다.
block_df["previous_feature_date"] = block_df.groupby(ID_COL)[FEATURE_DATE_COL].shift(1)
block_df["feature_date_gap_days"] = (
    block_df[FEATURE_DATE_COL] - block_df["previous_feature_date"]
).dt.days

# 첫 row 또는 1일 초과 gap이면 새 contiguous block으로 봅니다.
block_df["starts_new_contiguous_block"] = (
    block_df["feature_date_gap_days"].isna()
    | (block_df["feature_date_gap_days"] != 1)
).astype(int)

block_df["contiguous_block_id"] = (
    block_df
    .groupby(ID_COL)["starts_new_contiguous_block"]
    .cumsum()
    .astype(int)
)

block_df["within_block_index"] = (
    block_df
    .groupby([ID_COL, "contiguous_block_id"])
    .cumcount()
    .astype(int)
)

block_df["contiguous_history_length"] = block_df["within_block_index"] + 1
block_df["has_full_3day_history"] = (block_df["contiguous_history_length"] >= 3).astype(int)
block_df["has_full_7day_history"] = (block_df["contiguous_history_length"] >= 7).astype(int)

rolling_source_features = [
    feature for feature in base_features
    if feature in block_df.columns and pd.api.types.is_numeric_dtype(block_df[feature])
]

ROLLING_WINDOWS = [3, 7]

new_feature_data = {}
created_block_features = []

grouped_block = block_df.groupby([ID_COL, "contiguous_block_id"], sort=False)

for window in ROLLING_WINDOWS:
    for feature in rolling_source_features:
        rolling_obj = grouped_block[feature].rolling(window=window, min_periods=1)

        mean_col = f"{feature}_block_roll{window}_mean"
        std_col = f"{feature}_block_roll{window}_std"
        min_col = f"{feature}_block_roll{window}_min"
        max_col = f"{feature}_block_roll{window}_max"

        new_feature_data[mean_col] = rolling_obj.mean().reset_index(level=[0, 1], drop=True)
        new_feature_data[std_col] = rolling_obj.std().reset_index(level=[0, 1], drop=True).fillna(0.0)
        new_feature_data[min_col] = rolling_obj.min().reset_index(level=[0, 1], drop=True)
        new_feature_data[max_col] = rolling_obj.max().reset_index(level=[0, 1], drop=True)

        created_block_features.extend([mean_col, std_col, min_col, max_col])

for feature in rolling_source_features:
    diff1_col = f"{feature}_block_diff1"
    new_feature_data[diff1_col] = grouped_block[feature].diff(1).fillna(0.0)
    created_block_features.append(diff1_col)

for window in ROLLING_WINDOWS:
    for feature in rolling_source_features:
        mean_col = f"{feature}_block_roll{window}_mean"
        dev_col = f"{feature}_dev_from_block_roll{window}_mean"
        new_feature_data[dev_col] = block_df[feature] - new_feature_data[mean_col]
        created_block_features.append(dev_col)

# Optional calendar features: target date 기준 calendar는 미래 정보가 아니라 예측 시점에 알려진 정보로 볼 수 있습니다.
# 일단 후보로만 만들고, 다음 단계에서 포함/제외 비교가 가능하게 분리합니다.
calendar_feature_data = {
    "target_dayofweek_sin": np.sin(2 * np.pi * block_df[DATE_COL].dt.dayofweek / 7),
    "target_dayofweek_cos": np.cos(2 * np.pi * block_df[DATE_COL].dt.dayofweek / 7),
    "target_month_sin": np.sin(2 * np.pi * block_df[DATE_COL].dt.month / 12),
    "target_month_cos": np.cos(2 * np.pi * block_df[DATE_COL].dt.month / 12),
}
calendar_features = list(calendar_feature_data.keys())

engineered_block_features_df = pd.DataFrame(new_feature_data, index=block_df.index)
calendar_features_df = pd.DataFrame(calendar_feature_data, index=block_df.index)

block_engineered_df = pd.concat(
    [block_df, engineered_block_features_df, calendar_features_df],
    axis=1,
)

candidate_features_no_calendar = base_features + created_block_features
candidate_features_with_calendar = candidate_features_no_calendar + calendar_features

print("=== Block-Aware Feature Engineering Summary ===")
print("block_engineered_df shape:", block_engineered_df.shape)
print("base daytime features:", len(base_features))
print("created block-aware rolling/trend features:", len(created_block_features))
print("calendar candidate features:", len(calendar_features))
print("candidate features without calendar:", len(candidate_features_no_calendar))
print("candidate features with calendar:", len(candidate_features_with_calendar))

print("\n=== Contiguous Block Summary ===")
block_summary = (
    block_engineered_df
    .groupby([ID_COL, "contiguous_block_id"])
    .agg(
        rows=(TARGET_COL, "size"),
        split=(SPLIT_COL, "first"),
        min_feature_date=(FEATURE_DATE_COL, "min"),
        max_feature_date=(FEATURE_DATE_COL, "max"),
        target_mean=(TARGET_COL, "mean"),
    )
    .reset_index()
)

display(
    block_summary
    .groupby("split")
    .agg(
        blocks=("contiguous_block_id", "size"),
        rows=("rows", "sum"),
        mean_block_rows=("rows", "mean"),
        median_block_rows=("rows", "median"),
        max_block_rows=("rows", "max"),
        blocks_ge_3=("rows", lambda s: int((s >= 3).sum())),
        blocks_ge_7=("rows", lambda s: int((s >= 7).sum())),
        blocks_ge_14=("rows", lambda s: int((s >= 14).sum())),
    )
    .reset_index()
)

print("\n=== Gap Distribution Recheck ===")
display(
    block_engineered_df["feature_date_gap_days"]
    .fillna(0)
    .astype(int)
    .value_counts()
    .sort_index()
    .rename("rows")
    .reset_index()
    .rename(columns={"index": "feature_date_gap_days"})
)

print("\n=== Missing / Inf Check ===")
missing_counts = (
    block_engineered_df[candidate_features_with_calendar]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "feature"})
)
display(missing_counts.head(30))
print("features with any missing:", int((missing_counts["missing_count"] > 0).sum()))

inf_count = np.isinf(
    block_engineered_df[candidate_features_with_calendar].to_numpy(dtype=float)
).sum()
print("Inf count:", int(inf_count))

print("\n=== Row Availability For MLP Current-Day Candidate ===")
display(
    block_engineered_df
    .groupby(SPLIT_COL)
    .agg(
        rows=(TARGET_COL, "size"),
        participants=(ID_COL, "nunique"),
        target_mean=(TARGET_COL, "mean"),
        rows_with_full_3day_history=("has_full_3day_history", "sum"),
        rows_with_full_7day_history=("has_full_7day_history", "sum"),
    )
    .reset_index()
)

print("\n=== Contiguous Sequence Candidate Counts For GRU ===")
sequence_count_rows = []

for window in [7, 14]:
    for split_name, split_df in block_engineered_df.groupby(SPLIT_COL):
        samples = 0
        participants = set()

        for (participant_id, block_id), group in split_df.groupby([ID_COL, "contiguous_block_id"]):
            n_rows = len(group)
            if n_rows >= window:
                samples += n_rows - window + 1
                participants.add(participant_id)

        sequence_count_rows.append(
            {
                "window": window,
                "split": split_name,
                "candidate_sequence_samples": samples,
                "participants_with_sequences": len(participants),
            }
        )

display(pd.DataFrame(sequence_count_rows).sort_values(["window", "split"]).reset_index(drop=True))

print("\nReady for next step: choose calendar inclusion and create scaled tensors for MLP current-day and GRU.")

=== Block-Aware Feature Engineering Summary ===
block_engineered_df shape: (3116, 424)
base daytime features: 19
created block-aware rolling/trend features: 209
calendar candidate features: 4
candidate features without calendar: 228
candidate features with calendar: 232

=== Contiguous Block Summary ===


,split,blocks,rows,mean_block_rows,median_block_rows,max_block_rows,blocks_ge_3,blocks_ge_7,blocks_ge_14
0,test,64,524,8.187500,3.0,75,42,18,8
1,train,204,2135,10.465686,5.0,87,143,77,46
2,validation,51,457,8.960784,5.0,59,36,21,9



=== Gap Distribution Recheck ===


,feature_date_gap_days,rows
0,0,63
1,1,2797
2,3,142
3,4,28
4,5,21
5,6,8
6,7,11
7,8,6
8,9,4
9,10,3



=== Missing / Inf Check ===


...

features with any missing: 0
Inf count: 0

=== Row Availability For MLP Current-Day Candidate ===


,deep_learning_split,rows,participants,target_mean,rows_with_full_3day_history,rows_with_full_7day_history
0,test,524,12,0.379771,407,296
1,train,2135,41,0.405621,1763,1303
2,validation,457,10,0.393873,364,245



=== Contiguous Sequence Candidate Counts For GRU ===


,window,split,candidate_sequence_samples,participants_with_sequences
0,7,test,296,9
1,7,train,1303,33
2,7,validation,245,9
3,14,test,206,5
4,14,train,887,27
5,14,validation,136,7



Ready for next step: choose calendar inclusion and create scaled tensors for MLP current-day and GRU.


In [5]:
# Cell 4. train-only scaling 후 MLP current-day / GRU tensor 후보를 in-memory로 생성한다.
# 원하는 결과:
# - feature engineering된 previous-day daytime_only 후보 feature를 train split 기준으로만 scaling한다.
# - MLP current-day 입력: 각 target row의 engineered feature vector.
# - GRU 입력: contiguous block 안에서 window 7/14 sequence를 만든다.
# - 아직 파일 저장은 하지 않고 shape, class balance, NaN/Inf 여부만 확인한다.

from sklearn.preprocessing import StandardScaler

ID_COL = "participant_object_id"
DATE_COL = "calendar_date"
FEATURE_DATE_COL = "feature_date"
TARGET_COL = "good_sleep_label"
SPLIT_COL = "deep_learning_split"

# 우선 calendar는 제외한 228개 feature를 기본 후보로 사용합니다.
# calendar feature는 나중에 ablation으로 따로 비교하는 편이 해석이 깔끔합니다.
USE_CALENDAR_FEATURES = False

if USE_CALENDAR_FEATURES:
    tensor_feature_names = list(candidate_features_with_calendar)
else:
    tensor_feature_names = list(candidate_features_no_calendar)

tensor_df = block_engineered_df.copy()
tensor_df = tensor_df.sort_values([ID_COL, FEATURE_DATE_COL, DATE_COL]).reset_index(drop=True)

print("=== Tensor Feature Configuration ===")
print("USE_CALENDAR_FEATURES:", USE_CALENDAR_FEATURES)
print("tensor feature count:", len(tensor_feature_names))
print("rows:", len(tensor_df))

# train split에만 fit해서 validation/test 정보가 scaler에 새지 않도록 합니다.
train_mask = tensor_df[SPLIT_COL] == "train"
scaler = StandardScaler()
scaler.fit(tensor_df.loc[train_mask, tensor_feature_names].to_numpy(dtype=np.float32))

scaled_feature_array = scaler.transform(
    tensor_df[tensor_feature_names].to_numpy(dtype=np.float32)
).astype(np.float32)

scaled_feature_df = pd.DataFrame(
    scaled_feature_array,
    columns=tensor_feature_names,
    index=tensor_df.index,
)

print("\n=== Scaled Feature Check ===")
print("scaled_feature_array shape:", scaled_feature_array.shape)
print("NaN count:", int(np.isnan(scaled_feature_array).sum()))
print("Inf count:", int(np.isinf(scaled_feature_array).sum()))

train_scaled = scaled_feature_df.loc[train_mask, tensor_feature_names]
scale_check = pd.DataFrame(
    {
        "feature": tensor_feature_names,
        "train_scaled_mean": train_scaled.mean(axis=0).to_numpy(),
        "train_scaled_std": train_scaled.std(axis=0, ddof=0).to_numpy(),
    }
)

display(scale_check.head(20))
print("max abs train scaled mean:", float(np.abs(scale_check["train_scaled_mean"]).max()))
print("min train scaled std:", float(scale_check["train_scaled_std"].min()))
print("max train scaled std:", float(scale_check["train_scaled_std"].max()))

# MLP current-day tensor 후보: row 전체를 사용한다.
mlp_tensors = {}

for split_name, split_df in tensor_df.groupby(SPLIT_COL, sort=False):
    split_index = split_df.index.to_numpy()

    X = scaled_feature_array[split_index]
    y = split_df[TARGET_COL].to_numpy(dtype=np.int64)

    mlp_tensors[split_name] = {
        "X_mlp_current_day": X,
        "y": y,
        "participant_object_id": split_df[ID_COL].astype(str).to_numpy(),
        "calendar_date": split_df[DATE_COL].dt.strftime("%Y-%m-%d").to_numpy(),
        "feature_date": split_df[FEATURE_DATE_COL].dt.strftime("%Y-%m-%d").to_numpy(),
    }

mlp_summary_rows = []
for split_name, item in mlp_tensors.items():
    y = item["y"]
    mlp_summary_rows.append(
        {
            "split": split_name,
            "samples": len(y),
            "features": item["X_mlp_current_day"].shape[1],
            "participants": len(set(item["participant_object_id"])),
            "target_0": int((y == 0).sum()),
            "target_1": int((y == 1).sum()),
            "target_mean": float(y.mean()),
            "nan_count": int(np.isnan(item["X_mlp_current_day"]).sum()),
            "inf_count": int(np.isinf(item["X_mlp_current_day"]).sum()),
        }
    )

print("\n=== MLP Current-Day Tensor Candidate Summary ===")
display(pd.DataFrame(mlp_summary_rows).sort_values("split").reset_index(drop=True))

# GRU sequence tensor 후보: contiguous block 안에서만 window를 만든다.
def build_sequence_tensors_from_blocks(source_df, scaled_array, feature_names, window):
    X_list = []
    y_list = []
    participant_list = []
    window_start_date_list = []
    window_end_date_list = []
    target_date_list = []
    feature_date_list = []
    block_id_list = []

    for (participant_id, block_id), group in source_df.groupby([ID_COL, "contiguous_block_id"], sort=False):
        group = group.sort_values([FEATURE_DATE_COL, DATE_COL])
        idx = group.index.to_numpy()

        if len(idx) < window:
            continue

        for end_pos in range(window - 1, len(idx)):
            start_pos = end_pos - window + 1
            window_idx = idx[start_pos : end_pos + 1]
            target_row = group.iloc[end_pos]

            X_list.append(scaled_array[window_idx])
            y_list.append(int(target_row[TARGET_COL]))
            participant_list.append(str(participant_id))
            window_start_date_list.append(group.iloc[start_pos][FEATURE_DATE_COL].strftime("%Y-%m-%d"))
            window_end_date_list.append(group.iloc[end_pos][FEATURE_DATE_COL].strftime("%Y-%m-%d"))
            target_date_list.append(target_row[DATE_COL].strftime("%Y-%m-%d"))
            feature_date_list.append(target_row[FEATURE_DATE_COL].strftime("%Y-%m-%d"))
            block_id_list.append(int(block_id))

    if X_list:
        X_sequence = np.stack(X_list).astype(np.float32)
        y = np.asarray(y_list, dtype=np.int64)
    else:
        X_sequence = np.empty((0, window, len(feature_names)), dtype=np.float32)
        y = np.empty((0,), dtype=np.int64)

    return {
        "X_sequence": X_sequence,
        "y": y,
        "participant_object_id": np.asarray(participant_list),
        "window_start_feature_date": np.asarray(window_start_date_list),
        "window_end_feature_date": np.asarray(window_end_date_list),
        "target_date": np.asarray(target_date_list),
        "feature_date": np.asarray(feature_date_list),
        "contiguous_block_id": np.asarray(block_id_list, dtype=np.int64),
    }

sequence_tensors = {}

for window in [7, 14]:
    sequence_tensors[window] = {}

    for split_name, split_df in tensor_df.groupby(SPLIT_COL, sort=False):
        sequence_tensors[window][split_name] = build_sequence_tensors_from_blocks(
            source_df=split_df,
            scaled_array=scaled_feature_array,
            feature_names=tensor_feature_names,
            window=window,
        )

sequence_summary_rows = []
for window, split_items in sequence_tensors.items():
    for split_name, item in split_items.items():
        y = item["y"]
        X = item["X_sequence"]

        sequence_summary_rows.append(
            {
                "window": window,
                "split": split_name,
                "samples": len(y),
                "sequence_shape": str(tuple(X.shape)),
                "participants": len(set(item["participant_object_id"])),
                "target_0": int((y == 0).sum()) if len(y) else 0,
                "target_1": int((y == 1).sum()) if len(y) else 0,
                "target_mean": float(y.mean()) if len(y) else np.nan,
                "nan_count": int(np.isnan(X).sum()),
                "inf_count": int(np.isinf(X).sum()),
            }
        )

print("\n=== GRU Sequence Tensor Candidate Summary ===")
display(
    pd.DataFrame(sequence_summary_rows)
    .sort_values(["window", "split"])
    .reset_index(drop=True)
)

# 기존 previous-day sequence tensor sample 수와 맞는지 확인하기 위한 기대값입니다.
expected_previous_day_counts = pd.DataFrame(
    [
        {"window": 7, "split": "train", "expected_samples": 1303},
        {"window": 7, "split": "validation", "expected_samples": 245},
        {"window": 7, "split": "test", "expected_samples": 296},
        {"window": 14, "split": "train", "expected_samples": 887},
        {"window": 14, "split": "validation", "expected_samples": 136},
        {"window": 14, "split": "test", "expected_samples": 206},
    ]
)

actual_sequence_counts = pd.DataFrame(sequence_summary_rows)[["window", "split", "samples"]]
sequence_count_check = expected_previous_day_counts.merge(
    actual_sequence_counts,
    on=["window", "split"],
    how="left",
)
sequence_count_check["matches_expected"] = (
    sequence_count_check["expected_samples"] == sequence_count_check["samples"]
)

print("\n=== Sequence Count Check Against Previous-Day Baseline ===")
display(sequence_count_check.sort_values(["window", "split"]).reset_index(drop=True))

print("\nReady for next step: save engineered dataset/tensors and create a small experiment grid.")

=== Tensor Feature Configuration ===
USE_CALENDAR_FEATURES: False
tensor feature count: 228
rows: 3116

=== Scaled Feature Check ===
scaled_feature_array shape: (3116, 228)
NaN count: 0
Inf count: 0


,feature,train_scaled_mean,train_scaled_std
0,resting_hr_resting_heart_rate_mean,-3.573487e-09,1.0
1,resting_hr_error_mean,0.000000e+00,1.0
2,resting_hr_record_count,1.429395e-08,1.0
3,lightly_active_minutes_sum_missing_ind,0.000000e+00,0.0
4,lightly_active_minutes_sum,-1.072046e-08,1.0
5,moderately_active_minutes_sum_missing_ind,0.000000e+00,0.0
6,moderately_active_minutes_sum,5.360230e-09,1.0
7,sedentary_minutes_sum_missing_ind,0.000000e+00,0.0
8,sedentary_minutes_sum,7.146974e-09,1.0
9,very_active_minutes_sum_missing_ind,0.000000e+00,0.0


max abs train scaled mean: 2.858789471815726e-08
min train scaled std: 0.0
max train scaled std: 1.0

=== MLP Current-Day Tensor Candidate Summary ===


,split,samples,features,participants,target_0,target_1,target_mean,nan_count,inf_count
0,test,524,228,12,325,199,0.379771,0,0
1,train,2135,228,41,1269,866,0.405621,0,0
2,validation,457,228,10,277,180,0.393873,0,0



=== GRU Sequence Tensor Candidate Summary ===


,window,split,samples,sequence_shape,participants,target_0,target_1,target_mean,nan_count,inf_count
0,7,test,296,"(296, 7, 228)",9,173,123,0.415541,0,0
1,7,train,1303,"(1303, 7, 228)",33,779,524,0.402149,0,0
2,7,validation,245,"(245, 7, 228)",9,153,92,0.375510,0,0
3,14,test,206,"(206, 14, 228)",5,109,97,0.470874,0,0
4,14,train,887,"(887, 14, 228)",27,553,334,0.376550,0,0
5,14,validation,136,"(136, 14, 228)",7,88,48,0.352941,0,0



=== Sequence Count Check Against Previous-Day Baseline ===


,window,split,expected_samples,samples,matches_expected
0,7,test,296,296,True
1,7,train,1303,1303,True
2,7,validation,245,245,True
3,14,test,206,206,True
4,14,train,887,887,True
5,14,validation,136,136,True



Ready for next step: save engineered dataset/tensors and create a small experiment grid.


In [6]:
# Cell 5. train 기준 zero-variance feature를 확인하고 제거한 최종 feature 후보를 만든다.
# 원하는 결과:
# - train split에서 분산이 0인 feature를 찾는다.
# - 정보가 없는 constant feature를 제외한 최종 feature list를 만든다.
# - 제거 후 다시 train-only scaling을 적용하고 MLP/GRU tensor shape를 재확인한다.
# - 아직 파일 저장은 하지 않는다.

ID_COL = "participant_object_id"
DATE_COL = "calendar_date"
FEATURE_DATE_COL = "feature_date"
TARGET_COL = "good_sleep_label"
SPLIT_COL = "deep_learning_split"

VARIANCE_EPS = 1e-12

train_mask = tensor_df[SPLIT_COL] == "train"

train_raw_feature_df = tensor_df.loc[train_mask, tensor_feature_names]
train_feature_variance = train_raw_feature_df.var(axis=0, ddof=0)

zero_variance_features = (
    train_feature_variance[train_feature_variance <= VARIANCE_EPS]
    .sort_index()
    .index
    .tolist()
)

final_tensor_feature_names = [
    feature for feature in tensor_feature_names
    if feature not in zero_variance_features
]

print("=== Zero-Variance Feature Check ===")
print("input tensor features:", len(tensor_feature_names))
print("zero-variance features on train:", len(zero_variance_features))
print("final tensor features:", len(final_tensor_feature_names))

if zero_variance_features:
    display(
        pd.DataFrame(
            {
                "zero_variance_feature": zero_variance_features,
                "train_variance": train_feature_variance.loc[zero_variance_features].to_numpy(),
            }
        )
    )

# zero-variance feature 제거 후 scaler를 다시 fit합니다.
final_scaler = StandardScaler()
final_scaler.fit(tensor_df.loc[train_mask, final_tensor_feature_names].to_numpy(dtype=np.float32))

final_scaled_feature_array = final_scaler.transform(
    tensor_df[final_tensor_feature_names].to_numpy(dtype=np.float32)
).astype(np.float32)

print("\n=== Final Scaled Feature Check ===")
print("final_scaled_feature_array shape:", final_scaled_feature_array.shape)
print("NaN count:", int(np.isnan(final_scaled_feature_array).sum()))
print("Inf count:", int(np.isinf(final_scaled_feature_array).sum()))

final_train_scaled = final_scaled_feature_array[train_mask.to_numpy()]
final_train_scaled_mean = final_train_scaled.mean(axis=0)
final_train_scaled_std = final_train_scaled.std(axis=0, ddof=0)

print("max abs train scaled mean:", float(np.abs(final_train_scaled_mean).max()))
print("min train scaled std:", float(final_train_scaled_std.min()))
print("max train scaled std:", float(final_train_scaled_std.max()))

# MLP current-day tensor를 final feature 기준으로 다시 만든다.
final_mlp_tensors = {}

for split_name, split_df in tensor_df.groupby(SPLIT_COL, sort=False):
    split_index = split_df.index.to_numpy()

    X = final_scaled_feature_array[split_index]
    y = split_df[TARGET_COL].to_numpy(dtype=np.int64)

    final_mlp_tensors[split_name] = {
        "X_mlp_current_day": X,
        "y": y,
        "participant_object_id": split_df[ID_COL].astype(str).to_numpy(),
        "calendar_date": split_df[DATE_COL].dt.strftime("%Y-%m-%d").to_numpy(),
        "feature_date": split_df[FEATURE_DATE_COL].dt.strftime("%Y-%m-%d").to_numpy(),
    }

final_mlp_summary_rows = []
for split_name, item in final_mlp_tensors.items():
    y = item["y"]
    X = item["X_mlp_current_day"]

    final_mlp_summary_rows.append(
        {
            "split": split_name,
            "samples": len(y),
            "features": X.shape[1],
            "participants": len(set(item["participant_object_id"])),
            "target_0": int((y == 0).sum()),
            "target_1": int((y == 1).sum()),
            "target_mean": float(y.mean()),
            "nan_count": int(np.isnan(X).sum()),
            "inf_count": int(np.isinf(X).sum()),
        }
    )

print("\n=== Final MLP Current-Day Tensor Summary ===")
display(pd.DataFrame(final_mlp_summary_rows).sort_values("split").reset_index(drop=True))

# GRU sequence tensor도 final feature 기준으로 다시 만든다.
final_sequence_tensors = {}

for window in [7, 14]:
    final_sequence_tensors[window] = {}

    for split_name, split_df in tensor_df.groupby(SPLIT_COL, sort=False):
        final_sequence_tensors[window][split_name] = build_sequence_tensors_from_blocks(
            source_df=split_df,
            scaled_array=final_scaled_feature_array,
            feature_names=final_tensor_feature_names,
            window=window,
        )

final_sequence_summary_rows = []
for window, split_items in final_sequence_tensors.items():
    for split_name, item in split_items.items():
        y = item["y"]
        X = item["X_sequence"]

        final_sequence_summary_rows.append(
            {
                "window": window,
                "split": split_name,
                "samples": len(y),
                "sequence_shape": str(tuple(X.shape)),
                "participants": len(set(item["participant_object_id"])),
                "target_0": int((y == 0).sum()) if len(y) else 0,
                "target_1": int((y == 1).sum()) if len(y) else 0,
                "target_mean": float(y.mean()) if len(y) else np.nan,
                "nan_count": int(np.isnan(X).sum()),
                "inf_count": int(np.isinf(X).sum()),
            }
        )

print("\n=== Final GRU Sequence Tensor Summary ===")
display(
    pd.DataFrame(final_sequence_summary_rows)
    .sort_values(["window", "split"])
    .reset_index(drop=True)
)

final_actual_sequence_counts = pd.DataFrame(final_sequence_summary_rows)[["window", "split", "samples"]]
final_sequence_count_check = expected_previous_day_counts.merge(
    final_actual_sequence_counts,
    on=["window", "split"],
    how="left",
)
final_sequence_count_check["matches_expected"] = (
    final_sequence_count_check["expected_samples"] == final_sequence_count_check["samples"]
)

print("\n=== Final Sequence Count Check ===")
display(final_sequence_count_check.sort_values(["window", "split"]).reset_index(drop=True))

print("\nReady for next step: save final engineered previous-day rolling/trend tensors.")

=== Zero-Variance Feature Check ===
input tensor features: 228
zero-variance features on train: 106
final tensor features: 122


,zero_variance_feature,train_variance
0,calories_record_count_block_diff1,0.0
1,calories_record_count_block_roll3_std,0.0
2,calories_record_count_block_roll7_std,0.0
3,calories_record_count_dev_from_block_roll3_mean,0.0
4,calories_record_count_dev_from_block_roll7_mean,0.0
5,calories_record_count_missing_ind,0.0
6,calories_record_count_missing_ind_block_diff1,0.0
7,calories_record_count_missing_ind_block_roll3_max,0.0
8,calories_record_count_missing_ind_block_roll3_...,0.0
9,calories_record_count_missing_ind_block_roll3_min,0.0



=== Final Scaled Feature Check ===
final_scaled_feature_array shape: (3116, 122)
NaN count: 0
Inf count: 0
max abs train scaled mean: 7.097768843777885e-07
min train scaled std: 0.9999592304229736
max train scaled std: 1.0000085830688477

=== Final MLP Current-Day Tensor Summary ===


,split,samples,features,participants,target_0,target_1,target_mean,nan_count,inf_count
0,test,524,122,12,325,199,0.379771,0,0
1,train,2135,122,41,1269,866,0.405621,0,0
2,validation,457,122,10,277,180,0.393873,0,0



=== Final GRU Sequence Tensor Summary ===


,window,split,samples,sequence_shape,participants,target_0,target_1,target_mean,nan_count,inf_count
0,7,test,296,"(296, 7, 122)",9,173,123,0.415541,0,0
1,7,train,1303,"(1303, 7, 122)",33,779,524,0.402149,0,0
2,7,validation,245,"(245, 7, 122)",9,153,92,0.375510,0,0
3,14,test,206,"(206, 14, 122)",5,109,97,0.470874,0,0
4,14,train,887,"(887, 14, 122)",27,553,334,0.376550,0,0
5,14,validation,136,"(136, 14, 122)",7,88,48,0.352941,0,0



=== Final Sequence Count Check ===


,window,split,expected_samples,samples,matches_expected
0,7,test,296,296,True
1,7,train,1303,1303,True
2,7,validation,245,245,True
3,14,test,206,206,True
4,14,train,887,887,True
5,14,validation,136,136,True



Ready for next step: save final engineered previous-day rolling/trend tensors.


In [7]:
# Cell 6. 최종 previous-day rolling/trend daytime_only dataset과 tensor를 저장한다.
# 원하는 결과:
# - rolling/trend feature가 추가된 tabular dataset을 저장한다.
# - train-only scaler, 최종 feature 목록, zero-variance 제거 목록, metadata를 저장한다.
# - MLP current-day tensor와 GRU sequence tensor(window 7/14)를 저장한다.
# - 저장 후 파일 존재 여부와 핵심 shape를 다시 확인한다.

import json
from datetime import datetime

import joblib

OUTPUT_DIR = PREVIOUS_DAY_DIR / "rolling_trend_daytime_only"
MLP_DIR = OUTPUT_DIR / "mlp_current_day"
SEQUENCE_DIR = OUTPUT_DIR / "sequence_gru_ready"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MLP_DIR.mkdir(parents=True, exist_ok=True)
SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)

ENGINEERED_DATA_PATH = OUTPUT_DIR / "modeling_dataset_previous_day_daytime_only_rolling_trend.csv"
FINAL_FEATURE_PATH = OUTPUT_DIR / "rolling_trend_daytime_only_feature_columns.csv"
ZERO_VARIANCE_FEATURE_PATH = OUTPUT_DIR / "zero_variance_removed_features.csv"
SCALER_PATH = OUTPUT_DIR / "rolling_trend_daytime_only_standard_scaler.joblib"
METADATA_PATH = OUTPUT_DIR / "metadata.json"
TENSOR_SUMMARY_PATH = OUTPUT_DIR / "tensor_summary.csv"

# CSV에 저장할 dataset은 id/date/target/split/block 진단 컬럼 + 최종 feature를 포함합니다.
dataset_columns = [
    ID_COL,
    DATE_COL,
    FEATURE_DATE_COL,
    "feature_lag_days",
    TARGET_COL,
    SPLIT_COL,
    "contiguous_block_id",
    "within_block_index",
    "contiguous_history_length",
    "has_full_3day_history",
    "has_full_7day_history",
    "feature_date_gap_days",
] + final_tensor_feature_names

dataset_columns = [col for col in dataset_columns if col in block_engineered_df.columns]

save_df = block_engineered_df[dataset_columns].copy()
save_df.to_csv(ENGINEERED_DATA_PATH, index=False, encoding="utf-8-sig")

pd.DataFrame({"feature": final_tensor_feature_names}).to_csv(
    FINAL_FEATURE_PATH,
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame({"removed_feature": zero_variance_features}).to_csv(
    ZERO_VARIANCE_FEATURE_PATH,
    index=False,
    encoding="utf-8-sig",
)

joblib.dump(final_scaler, SCALER_PATH)

# MLP tensors 저장
for split_name, item in final_mlp_tensors.items():
    split_path = MLP_DIR / f"{split_name}.npz"

    np.savez_compressed(
        split_path,
        X_mlp_current_day=item["X_mlp_current_day"].astype(np.float32),
        y=item["y"].astype(np.int64),
        participant_object_id=item["participant_object_id"],
        calendar_date=item["calendar_date"],
        feature_date=item["feature_date"],
        feature_names=np.asarray(final_tensor_feature_names),
    )

# GRU-ready sequence tensors 저장
for window, split_items in final_sequence_tensors.items():
    window_dir = SEQUENCE_DIR / f"window_{window}"
    window_dir.mkdir(parents=True, exist_ok=True)

    for split_name, item in split_items.items():
        split_path = window_dir / f"{split_name}.npz"

        np.savez_compressed(
            split_path,
            X_sequence=item["X_sequence"].astype(np.float32),
            y=item["y"].astype(np.int64),
            participant_object_id=item["participant_object_id"],
            window_start_feature_date=item["window_start_feature_date"],
            window_end_feature_date=item["window_end_feature_date"],
            target_date=item["target_date"],
            feature_date=item["feature_date"],
            contiguous_block_id=item["contiguous_block_id"],
            feature_names=np.asarray(final_tensor_feature_names),
        )

# summary 저장
summary_rows = []

for split_name, item in final_mlp_tensors.items():
    X = item["X_mlp_current_day"]
    y = item["y"]

    summary_rows.append(
        {
            "tensor_type": "mlp_current_day",
            "window": 1,
            "split": split_name,
            "samples": len(y),
            "participants": len(set(item["participant_object_id"])),
            "features": X.shape[1],
            "shape": str(tuple(X.shape)),
            "target_0": int((y == 0).sum()),
            "target_1": int((y == 1).sum()),
            "target_mean": float(y.mean()),
            "nan_count": int(np.isnan(X).sum()),
            "inf_count": int(np.isinf(X).sum()),
        }
    )

for window, split_items in final_sequence_tensors.items():
    for split_name, item in split_items.items():
        X = item["X_sequence"]
        y = item["y"]

        summary_rows.append(
            {
                "tensor_type": "gru_sequence",
                "window": window,
                "split": split_name,
                "samples": len(y),
                "participants": len(set(item["participant_object_id"])),
                "features": X.shape[2] if X.ndim == 3 else len(final_tensor_feature_names),
                "shape": str(tuple(X.shape)),
                "target_0": int((y == 0).sum()) if len(y) else 0,
                "target_1": int((y == 1).sum()) if len(y) else 0,
                "target_mean": float(y.mean()) if len(y) else np.nan,
                "nan_count": int(np.isnan(X).sum()),
                "inf_count": int(np.isinf(X).sum()),
            }
        )

tensor_summary = pd.DataFrame(summary_rows)
tensor_summary.to_csv(TENSOR_SUMMARY_PATH, index=False, encoding="utf-8-sig")

metadata = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_dataset": str(PREVIOUS_DAY_DATA_PATH.relative_to(PROJECT_ROOT)),
    "source_feature_subset": str(DAYTIME_ONLY_FEATURE_PATH.relative_to(PROJECT_ROOT)),
    "feature_timing": "previous_day",
    "subset_name": "daytime_only_rolling_trend",
    "base_feature_count": len(base_features),
    "engineered_feature_count_before_zero_variance_filter": len(tensor_feature_names),
    "zero_variance_removed_count": len(zero_variance_features),
    "final_feature_count": len(final_tensor_feature_names),
    "rolling_windows": ROLLING_WINDOWS,
    "rolling_statistics": ["mean", "std", "min", "max"],
    "trend_features": ["block_diff1", "dev_from_block_roll3_mean", "dev_from_block_roll7_mean"],
    "calendar_features_used": bool(USE_CALENDAR_FEATURES),
    "contiguous_block_rule": "new block when participant changes or feature_date gap is not exactly 1 day",
    "scaler_rule": "StandardScaler fit on train split only after zero-variance feature removal",
    "saved_outputs": {
        "engineered_dataset": str(ENGINEERED_DATA_PATH.relative_to(PROJECT_ROOT)),
        "feature_columns": str(FINAL_FEATURE_PATH.relative_to(PROJECT_ROOT)),
        "zero_variance_removed_features": str(ZERO_VARIANCE_FEATURE_PATH.relative_to(PROJECT_ROOT)),
        "scaler": str(SCALER_PATH.relative_to(PROJECT_ROOT)),
        "mlp_current_day_dir": str(MLP_DIR.relative_to(PROJECT_ROOT)),
        "sequence_gru_ready_dir": str(SEQUENCE_DIR.relative_to(PROJECT_ROOT)),
        "tensor_summary": str(TENSOR_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
    },
}

METADATA_PATH.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("=== Saved Output Paths ===")
for path in [
    ENGINEERED_DATA_PATH,
    FINAL_FEATURE_PATH,
    ZERO_VARIANCE_FEATURE_PATH,
    SCALER_PATH,
    METADATA_PATH,
    TENSOR_SUMMARY_PATH,
]:
    print(path.relative_to(PROJECT_ROOT), "exists:", path.exists())

print("\n=== Saved Tensor Files ===")
saved_tensor_files = sorted(
    list(MLP_DIR.glob("*.npz"))
    + list(SEQUENCE_DIR.glob("window_*/*.npz"))
)
for path in saved_tensor_files:
    print(path.relative_to(PROJECT_ROOT), "exists:", path.exists())

print("\n=== Tensor Summary ===")
display(tensor_summary.sort_values(["tensor_type", "window", "split"]).reset_index(drop=True))

print("\nReady for next step: update log/2026-06-29.md and prepare experiment grid for MLP current-day + GRU.")

=== Saved Output Paths ===
data\processed\deep_learning_previous_day\rolling_trend_daytime_only\modeling_dataset_previous_day_daytime_only_rolling_trend.csv exists: True
data\processed\deep_learning_previous_day\rolling_trend_daytime_only\rolling_trend_daytime_only_feature_columns.csv exists: True
data\processed\deep_learning_previous_day\rolling_trend_daytime_only\zero_variance_removed_features.csv exists: True
data\processed\deep_learning_previous_day\rolling_trend_daytime_only\rolling_trend_daytime_only_standard_scaler.joblib exists: True
data\processed\deep_learning_previous_day\rolling_trend_daytime_only\metadata.json exists: True
data\processed\deep_learning_previous_day\rolling_trend_daytime_only\tensor_summary.csv exists: True

=== Saved Tensor Files ===
data\processed\deep_learning_previous_day\rolling_trend_daytime_only\mlp_current_day\test.npz exists: True
data\processed\deep_learning_previous_day\rolling_trend_daytime_only\mlp_current_day\train.npz exists: True
data\process

,tensor_type,window,split,samples,participants,features,shape,target_0,target_1,target_mean,nan_count,inf_count
0,gru_sequence,7,test,296,9,122,"(296, 7, 122)",173,123,0.415541,0,0
1,gru_sequence,7,train,1303,33,122,"(1303, 7, 122)",779,524,0.402149,0,0
2,gru_sequence,7,validation,245,9,122,"(245, 7, 122)",153,92,0.375510,0,0
3,gru_sequence,14,test,206,5,122,"(206, 14, 122)",109,97,0.470874,0,0
4,gru_sequence,14,train,887,27,122,"(887, 14, 122)",553,334,0.376550,0,0
5,gru_sequence,14,validation,136,7,122,"(136, 14, 122)",88,48,0.352941,0,0
6,mlp_current_day,1,test,524,12,122,"(524, 122)",325,199,0.379771,0,0
7,mlp_current_day,1,train,2135,41,122,"(2135, 122)",1269,866,0.405621,0,0
8,mlp_current_day,1,validation,457,10,122,"(457, 122)",277,180,0.393873,0,0



Ready for next step: update log/2026-06-29.md and prepare experiment grid for MLP current-day + GRU.


In [8]:
# Cell 7. work log 업데이트
# 원하는 결과:
# - 이번 08 노트북에서 생성한 rolling/trend feature engineering 산출물을 log/2026-06-29.md에 기록한다.
# - UTF-8로 append한다.
# - notebooks/02_pipeline_from_scripts_summary.ipynb는 아직 업데이트하지 않는다.

from datetime import datetime

LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

log_entry = f"""

## Previous-Day Rolling/Trend Daytime-Only Feature Engineering

### 1. Reason for update

- Started `notebooks/08_previous_day_rolling_trend_feature_engineering.ipynb`.
- The goal is to improve strict previous-day forecasting by adding history-aware features to the conservative `daytime_only` subset.
- Model training was not run.
- Logistic Regression and Random Forest remain traditional ML baseline/reference only.
- PCA was not used.

### 2. Input files

- `data/processed/deep_learning_previous_day/modeling_dataset_previous_day_encoded.csv`
- `data/processed/deep_learning_feature_subsets/daytime_only_features.csv`

### 3. Feature engineering design

- Used previous-day rows where `feature_date = target calendar_date - 1 day`.
- Focused on `daytime_only` features first.
- Created contiguous participant-level blocks so rolling history does not cross feature-date gaps.
- Created block-aware rolling features:
  - 3-day and 7-day rolling mean
  - 3-day and 7-day rolling std
  - 3-day and 7-day rolling min
  - 3-day and 7-day rolling max
- Created trend/difference features:
  - 1-row within-block difference
  - deviation from 3-day rolling mean
  - deviation from 7-day rolling mean
- Calendar features were prepared as optional candidates but were not included in this saved tensor set.

### 4. Created outputs

- Created:
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/modeling_dataset_previous_day_daytime_only_rolling_trend.csv`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/rolling_trend_daytime_only_feature_columns.csv`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/zero_variance_removed_features.csv`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/rolling_trend_daytime_only_standard_scaler.joblib`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/metadata.json`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/tensor_summary.csv`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/mlp_current_day/train.npz`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/mlp_current_day/validation.npz`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/mlp_current_day/test.npz`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/sequence_gru_ready/window_7/train.npz`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/sequence_gru_ready/window_7/validation.npz`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/sequence_gru_ready/window_7/test.npz`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/sequence_gru_ready/window_14/train.npz`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/sequence_gru_ready/window_14/validation.npz`
  - `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/sequence_gru_ready/window_14/test.npz`

### 5. Validation summary

- Previous-day source rows: `3116`
- All rows had valid 1-day lag from `feature_date` to target `calendar_date`.
- Participant split overlap: `0`
- Base daytime-only features: `19`
- Engineered candidate features before zero-variance filtering: `228`
- Train zero-variance features removed: `106`
- Final saved feature count: `122`
- Train-only `StandardScaler` was fit after zero-variance feature removal.
- Saved tensors had no NaN or Inf values.

| tensor_type | window | train | validation | test | features |
| --- | ---: | ---: | ---: | ---: | ---: |
| `mlp_current_day` | 1 | 2135 | 457 | 524 | 122 |
| `gru_sequence` | 7 | 1303 | 245 | 296 | 122 |
| `gru_sequence` | 14 | 887 | 136 | 206 | 122 |

### 6. Next planned work

- Create a small rolling/trend experiment grid for:
  - `mlp_current_day`
  - `gru`
- Compare first against the previous-day conservative candidate:
  - `phase2a_006`
  - `previous_day / daytime_only / window 14 / BiLSTM`
  - test balanced accuracy `0.6098`
- Keep final interpretation scoped as strict previous-day forecasting only if validation/test results improve under this timing design.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("log updated:", LOG_PATH)
print("exists:", LOG_PATH.exists())
print("appended characters:", len(log_entry))

log updated: c:\workSpace\DeepLearnin_sleep\log\2026-06-29.md
exists: True
appended characters: 4164


In [9]:
# Cell 8. rolling/trend previous-day 실험 grid 생성
# 원하는 결과:
# - 새 tensor 산출물을 대상으로 MLP current-day와 GRU 우선 비교 grid를 만든다.
# - training은 실행하지 않는다.
# - grid CSV만 저장한다.

EXPERIMENT_DIR = PREVIOUS_DAY_DIR / "experiments" / "phase_3a_rolling_trend_daytime_only"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

GRID_PATH = EXPERIMENT_DIR / "phase_3a_rolling_trend_experiment_grid.csv"

experiment_rows = [
    {
        "experiment_id": "phase3a_000",
        "feature_timing": "previous_day",
        "subset_name": "daytime_only_rolling_trend",
        "model_family": "mlp_current_day",
        "window": 1,
        "tensor_dir": str((OUTPUT_DIR / "mlp_current_day").relative_to(PROJECT_ROOT)),
        "feature_count": len(final_tensor_feature_names),
        "calendar_features_used": bool(USE_CALENDAR_FEATURES),
        "notes": "MLP on previous-day daytime_only rolling/trend tabular features",
    },
    {
        "experiment_id": "phase3a_001",
        "feature_timing": "previous_day",
        "subset_name": "daytime_only_rolling_trend",
        "model_family": "gru",
        "window": 7,
        "tensor_dir": str((OUTPUT_DIR / "sequence_gru_ready" / "window_7").relative_to(PROJECT_ROOT)),
        "feature_count": len(final_tensor_feature_names),
        "calendar_features_used": bool(USE_CALENDAR_FEATURES),
        "notes": "GRU on previous-day daytime_only rolling/trend sequence features, window 7",
    },
    {
        "experiment_id": "phase3a_002",
        "feature_timing": "previous_day",
        "subset_name": "daytime_only_rolling_trend",
        "model_family": "gru",
        "window": 14,
        "tensor_dir": str((OUTPUT_DIR / "sequence_gru_ready" / "window_14").relative_to(PROJECT_ROOT)),
        "feature_count": len(final_tensor_feature_names),
        "calendar_features_used": bool(USE_CALENDAR_FEATURES),
        "notes": "GRU on previous-day daytime_only rolling/trend sequence features, window 14",
    },
]

phase3a_grid = pd.DataFrame(experiment_rows)
phase3a_grid.to_csv(GRID_PATH, index=False, encoding="utf-8-sig")

print("=== Phase 3A Rolling/Trend Experiment Grid ===")
print("grid path:", GRID_PATH.relative_to(PROJECT_ROOT))
print("exists:", GRID_PATH.exists())
display(phase3a_grid)

=== Phase 3A Rolling/Trend Experiment Grid ===
grid path: data\processed\deep_learning_previous_day\experiments\phase_3a_rolling_trend_daytime_only\phase_3a_rolling_trend_experiment_grid.csv
exists: True


,experiment_id,feature_timing,subset_name,model_family,window,tensor_dir,feature_count,calendar_features_used,notes
0,phase3a_000,previous_day,daytime_only_rolling_trend,mlp_current_day,1,data\processed\deep_learning_previous_day\roll...,122,False,MLP on previous-day daytime_only rolling/trend...
1,phase3a_001,previous_day,daytime_only_rolling_trend,gru,7,data\processed\deep_learning_previous_day\roll...,122,False,GRU on previous-day daytime_only rolling/trend...
2,phase3a_002,previous_day,daytime_only_rolling_trend,gru,14,data\processed\deep_learning_previous_day\roll...,122,False,GRU on previous-day daytime_only rolling/trend...


In [10]:
# Cell 9. Phase 3A experiment grid 생성 내역을 log에 추가한다.
# 원하는 결과:
# - phase_3a_rolling_trend_experiment_grid.csv 생성 사실을 log/2026-06-29.md에 기록한다.
# - training은 실행하지 않는다.

LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

grid_log_entry = """

## Phase 3A Rolling/Trend Experiment Grid

### 1. Reason for update

- Created a small first-pass experiment grid for the previous-day rolling/trend daytime-only tensor set.
- The goal is to compare whether engineered rolling/trend features improve strict previous-day forecasting before expanding to larger model grids.
- Model training was not run.

### 2. Created output

- Created:
  - `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_rolling_trend_experiment_grid.csv`

### 3. Experiment grid

| experiment_id | feature_timing | subset_name | model_family | window | feature_count |
| --- | --- | --- | --- | ---: | ---: |
| `phase3a_000` | `previous_day` | `daytime_only_rolling_trend` | `mlp_current_day` | 1 | 122 |
| `phase3a_001` | `previous_day` | `daytime_only_rolling_trend` | `gru` | 7 | 122 |
| `phase3a_002` | `previous_day` | `daytime_only_rolling_trend` | `gru` | 14 | 122 |

### 4. Next planned work

- Validate that the grid paths load the expected `.npz` tensors.
- Then run Phase 3A training manually when ready.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(grid_log_entry)

print("log updated:", LOG_PATH)
print("exists:", LOG_PATH.exists())
print("appended characters:", len(grid_log_entry))

log updated: c:\workSpace\DeepLearnin_sleep\log\2026-06-29.md
exists: True
appended characters: 1088


In [11]:
# Cell 10. Phase 3A grid와 저장된 tensor 파일 로드 검증
# 원하는 결과:
# - grid의 tensor_dir 경로가 모두 존재하는지 확인한다.
# - train/validation/test npz 파일이 모두 있는지 확인한다.
# - 각 experiment의 입력 shape와 target 분포를 확인한다.
# - training은 실행하지 않는다.

phase3a_grid = pd.read_csv(GRID_PATH, encoding="utf-8-sig")

validation_rows = []

for _, row in phase3a_grid.iterrows():
    experiment_id = row["experiment_id"]
    model_family = row["model_family"]
    window = int(row["window"])
    tensor_dir = PROJECT_ROOT / row["tensor_dir"]

    for split in ["train", "validation", "test"]:
        npz_path = tensor_dir / f"{split}.npz"
        exists = npz_path.exists()

        if exists:
            data = np.load(npz_path, allow_pickle=True)

            if model_family == "mlp_current_day":
                X = data["X_mlp_current_day"]
            else:
                X = data["X_sequence"]

            y = data["y"]

            validation_rows.append(
                {
                    "experiment_id": experiment_id,
                    "model_family": model_family,
                    "window": window,
                    "split": split,
                    "npz_exists": exists,
                    "X_shape": str(tuple(X.shape)),
                    "samples": len(y),
                    "target_0": int((y == 0).sum()),
                    "target_1": int((y == 1).sum()),
                    "target_mean": float(y.mean()),
                    "nan_count": int(np.isnan(X).sum()),
                    "inf_count": int(np.isinf(X).sum()),
                    "feature_count": len(data["feature_names"]),
                }
            )
        else:
            validation_rows.append(
                {
                    "experiment_id": experiment_id,
                    "model_family": model_family,
                    "window": window,
                    "split": split,
                    "npz_exists": exists,
                    "X_shape": None,
                    "samples": None,
                    "target_0": None,
                    "target_1": None,
                    "target_mean": None,
                    "nan_count": None,
                    "inf_count": None,
                    "feature_count": None,
                }
            )

phase3a_tensor_validation = pd.DataFrame(validation_rows)

print("=== Phase 3A Tensor Validation ===")
display(
    phase3a_tensor_validation
    .sort_values(["experiment_id", "split"])
    .reset_index(drop=True)
)

print("\nproblem rows:")
display(
    phase3a_tensor_validation[
        (phase3a_tensor_validation["npz_exists"] != True)
        | (phase3a_tensor_validation["nan_count"] != 0)
        | (phase3a_tensor_validation["inf_count"] != 0)
    ]
)

=== Phase 3A Tensor Validation ===


,experiment_id,model_family,window,split,npz_exists,X_shape,samples,target_0,target_1,target_mean,nan_count,inf_count,feature_count
0,phase3a_000,mlp_current_day,1,test,True,"(524, 122)",524,325,199,0.379771,0,0,122
1,phase3a_000,mlp_current_day,1,train,True,"(2135, 122)",2135,1269,866,0.405621,0,0,122
2,phase3a_000,mlp_current_day,1,validation,True,"(457, 122)",457,277,180,0.393873,0,0,122
3,phase3a_001,gru,7,test,True,"(296, 7, 122)",296,173,123,0.415541,0,0,122
4,phase3a_001,gru,7,train,True,"(1303, 7, 122)",1303,779,524,0.402149,0,0,122
5,phase3a_001,gru,7,validation,True,"(245, 7, 122)",245,153,92,0.375510,0,0,122
6,phase3a_002,gru,14,test,True,"(206, 14, 122)",206,109,97,0.470874,0,0,122
7,phase3a_002,gru,14,train,True,"(887, 14, 122)",887,553,334,0.376550,0,0,122
8,phase3a_002,gru,14,validation,True,"(136, 14, 122)",136,88,48,0.352941,0,0,122



problem rows:


,experiment_id,model_family,window,split,npz_exists,X_shape,samples,target_0,target_1,target_mean,nan_count,inf_count,feature_count
